In [1]:
import keras
class ResidualBlock(keras.layers.Layer):
    def __init__(self,filter,kernel,input_tensor,strides,activation='relu'):
        super().__init__()
        self.strides=strides
        self.input_tensor=input_tensor
        self.filter=filter
        self.kernel=kernel
        self.activation=activation
        self.conv1=keras.layers.Conv2D(filter,1,activation=activation,strides=strides,padding='same')
        self.bnl=keras.layers.BatchNormalization()
        self.conv2=keras.layers.Conv2D(filter,kernel,activation=activation,strides=1,padding='same')
        self.bnl1=keras.layers.BatchNormalization()
        self.projection=keras.layers.Conv2D(filter,1,activation=activation,strides=strides,padding='same')
        self.conv3=keras.layers.Conv2D(filters=(filter*4),kernel_size=1,strides=1,activation=activation,padding='same')
        self.bnl2=keras.layers.BatchNormalization()
    def call(self,input_tensor):
        x=self.conv1(input_tensor)
        x=self.bnl(x)
        x=self.conv2(x)
        x=self.bnl1(x)
        x=self.conv3(x)
        x=self.bnl2(x)
        x=keras.layers.add([x,input_tensor])
        return x
preprocessing_part=keras.Sequential([keras.layers.Resizing(128,128),keras.layers.Rescaling(1./255),keras.layers.CenterCrop(120,120)
,keras.layers.RandomFlip(),keras.layers.RandomZoom(0.1),keras.layers.RandomRotation(0.1)])

2026-09-23 16:19:43.484415: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790180384.125806 3209766 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1790180384.369286 3209766 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1790180386.364980 3209766 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790180386.365749 3209766 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790180386.365752 3209766 computation_placer.cc:177] computation placer alr

In [2]:
input=keras.Input(shape=(128,128,3))
x=preprocessing_part(input)
x=keras.layers.Conv2D(64,1,activation='relu',padding='same')(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Conv2D(64,3,activation='relu',padding='same')(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Conv2D(256,1,activation='relu',padding='same')(x)
stem_block=keras.layers.BatchNormalization()(x)

block1=ResidualBlock(filter=64,kernel=3,strides=1,input_tensor=stem_block)(stem_block)
block2=ResidualBlock(filter=64,kernel=3,strides=1,input_tensor=block1)(block1)
block3=ResidualBlock(filter=64,kernel=3,strides=1,input_tensor=block2)(block2)
x=keras.layers.Conv2D(128,1,strides=2,activation='relu',padding='same')(block3)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Conv2D(128,1,activation='relu',padding='same',strides=1)(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Conv2D(512,1,activation='relu',padding='same',strides=1)(x)
block4=keras.layers.BatchNormalization()(x)
block5=ResidualBlock(filter=128,kernel=3,strides=1,input_tensor=block4)(block4)
block6=ResidualBlock(filter=128,kernel=3,strides=1,input_tensor=block5)(block5)
block7=ResidualBlock(filter=128,kernel=3,strides=1,input_tensor=block6)(block6)
x=keras.layers.Conv2D(256,1,activation='relu',padding='same',strides=2)(block7)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Conv2D(256,1,activation='relu',padding='same')(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Conv2D(1024,1,activation='relu',padding='same')(x)
block8=keras.layers.BatchNormalization()(x)
block9=ResidualBlock(filter=256,kernel=3,strides=1,input_tensor=block8)(block8)
block10=ResidualBlock(filter=256,kernel=3,strides=1,input_tensor=block9)(block9)
block11=ResidualBlock(filter=256,kernel=3,strides=1,input_tensor=block10)(block10)
block12=ResidualBlock(filter=256,kernel=3,strides=1,input_tensor=block11)(block11)
block13=ResidualBlock(filter=256,kernel=3,strides=1,input_tensor=block12)(block12)
x=keras.layers.Conv2D(512,1,activation='relu',padding='same',strides=2)(block13)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Conv2D(512,1,activation='relu',padding='same')(x)
x=keras.layers.BatchNormalization()(x)
x=keras.layers.Conv2D(2048,1,activation='relu',padding='same')(x)
block14=keras.layers.BatchNormalization()(x)
block15=ResidualBlock(filter=512,kernel=3,strides=1,input_tensor=block14)(block14)
block16=ResidualBlock(filter=512,kernel=3,strides=1,input_tensor=block15)(block15)
x=keras.layers.Conv2D(512,3,activation='relu')(block16)
x=keras.layers.GlobalAveragePooling2D()(x)
x=keras.layers.Dense(512)(x)
x=keras.layers.Dropout(0.5)(x)
output=keras.layers.Dense(4,activation='softmax')(x)
resnet50_model=keras.Model(input,output)
resnet50_model.compile(optimizer=keras.optimizers.Adam(learning_rate=keras.optimizers.schedules.ExponentialDecay(0.0001, decay_steps=1000, decay_rate=0.9)),loss=keras.losses.SparseCategoricalCrossentropy(),metrics=['accuracy'])
resnet50_model.summary()

/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:395: UserWarning: `build()` was called on layer 'residual_block', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:395: UserWarning: `build()` was called on layer 'residual_block_1', however the layer does not have a `build()` method implemented and it looks like it has unbuilt state. This will cause the layer to be marked as built, despite not being actually built, which may cause failures down the line. Make sure to implement a proper `build()` method.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/keras/src/layers/layer.py:395: UserWarning: `build()` was called on layer 'residual_block_2', however the la

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 128, 128, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ sequential (Sequential)         │ (None, 120, 120, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d (Conv2D)                 │ (None, 120, 120, 64)   │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 120, 120, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 120, 120, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 120, 120, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 120, 120, 256)  │        16,640 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 120, 120, 256)  │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block (ResidualBlock)  │ (None, 120, 120, 256)  │        71,552 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_1                │ (None, 120, 120, 256)  │        71,552 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_2                │ (None, 120, 120, 256)  │        71,552 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_15 (Conv2D)              │ (None, 60, 60, 128)    │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_12          │ (None, 60, 60, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_16 (Conv2D)              │ (None, 60, 60, 128)    │        16,512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_13          │ (None, 60, 60, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_17 (Conv2D)              │ (None, 60, 60, 512)    │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_14          │ (None, 60, 60, 512)    │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_3                │ (None, 60, 60, 512)    │       282,368 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_4                │ (None, 60, 60, 512)    │       282,368 │
│ (ResidualBlock)                 │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ residual_block_5                │ (None, 60, 60, 512)    │       282,36

 Total params: 27,807,428 (106.08 MB)

 Trainable params: 27,761,348 (105.90 MB)

 Non-trainable params: 46,080 (180.00 KB)

In [ ]:
import tensorflow as tf
train_ds=keras.utils.image_dataset_from_directory("Training",batch_size=8,seed=123,validation_split=0.25,subset="training",
image_size=(128,128))
test_ds=keras.utils.image_dataset_from_directory("Training",batch_size=8,seed=456,validation_split=0.25,subset="validation",
image_size=(128,128))
train_ds.cache().prefetch(tf.data.AUTOTUNE)
test_ds.cache().prefetch(tf.data.AUTOTUNE)
resnet50_model.fit(train_ds,epochs=32,validation_data=test_ds,callbacks=[keras.callbacks.EarlyStopping(monitor='val_loss',restore_best_weights=True,start_from_epoch=16)])

In [5]:
test_data=keras.utils.image_dataset_from_directory("Testing",batch_size=8,seed=789,image_size=(128,128))
resnet50_model.evaluate(test_data)

Found 1600 files belonging to 4 classes.
200/200 ━━━━━━━━━━━━━━━━━━━━ 42s 207ms/step - accuracy: 0.8612 - loss: 0.6729


[0.6959478259086609, 0.8656250238418579]

In [9]:
import numpy as np
y_true=[]
for images,labels in test_ds:
    y_true.extend(labels.numpy())
y_true=np.array(y_true)
print(y_true.shape)
print(y_true[:40])
y_pred=resnet50_model.predict(test_ds)
y_pred=np.argmax(y_pred,axis=1)
print(y_pred.shape)
print(y_pred[:40])
print("Number of correct samples:",np.sum(y_true==y_pred))
total=len(y_true)
print("Total number of samples:",total)
print("Accuracy:",np.sum(y_true==y_pred)/total)

(1400,)
[0 1 2 0 2 3 1 1 3 2 0 3 0 0 0 2 3 0 0 2 2 2 1 3 1 0 2 0 0 3 0 2 3 3 2 0 3
 3 3 3]
175/175 ━━━━━━━━━━━━━━━━━━━━ 35s 206ms/step
(1400,)
[3 3 1 2 2 2 2 2 2 2 0 1 2 2 2 0 3 1 3 2 1 1 2 2 3 2 0 2 2 1 2 2 2 1 3 0 2
 2 1 3]
Number of correct samples: 333
Total number of samples: 1400
Accuracy: 0.23785714285714285
